## **<span style="color:red">- Model Selection</span>**

<br>

#### **<span style="color:black">- Dataset</span>**

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Linear Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Support Vector Machine
from sklearn.svm import SVR

# Tree-based Models
from sklearn.tree import DecisionTreeRegressor

# Ensemble Models
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)

# Neural Network
from sklearn.neural_network import MLPRegressor

# XGBoost (external library)
from xgboost import XGBRegressor

In [3]:
df = pd.read_csv('gurgaon_properties_post_feature_selection.csv')
df.shape

(3498, 13)

In [3]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,study room,servant room,store room,furnishing_type,floor_category
0,flat,sector 36,0.82,3,2,2,New Property,850.0,0,0,0,semifurnished,Low Floor
1,flat,sector 89,0.95,2,2,2,New Property,1226.0,1,1,0,semifurnished,Mid Floor
2,flat,sohna road,0.32,2,2,1,New Property,1000.0,0,0,0,semifurnished,High Floor
3,flat,sector 92,1.60,3,4,3+,Relatively New,1615.0,0,1,0,furnished,Mid Floor
4,flat,sector 102,0.48,2,2,1,Relatively New,581.0,0,0,1,semifurnished,Mid Floor


In [69]:
X = df.drop(columns=['price'])
y = df['price']

In [9]:
# Applying the log1p transformation to the target variable --> Handling Skewness (Normalizing the Distribution) (Bell Curve)
y_transformed = np.log1p(y)

<br>

#### **<span style="color:black">- Encoding Techniques</span>**

<br>

##### **<span style="color:green">1. Ordinal Encoding</span>**

<br>

###### **<span style="color:black">One Model</span>**

In [6]:
columns_to_encode = ['property_type', 'sector', 'balcony', 'agePossession', 'furnishing_type', 'floor_category']

In [ ]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room', 'study room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode)
    ], 
    remainder='passthrough'
)

In [11]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [12]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2',error_score='raise')
scores.mean(),scores.std()

(np.float64(0.7390239098525023), np.float64(0.021635908754284502))

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [14]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [15]:
y_pred = pipeline.predict(X_test)
y_pred = np.expm1(y_pred)

In [16]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.8853672918524242

<br>

###### **<span style="color:black">ALL Models</span>**

In [ ]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room', 'study room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode)
    ], 
    remainder='passthrough'
)

In [17]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [20]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [21]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [22]:
model_output

[['linear_reg', np.float64(0.7390239098525023), 0.8853672918524242],
 ['svr', np.float64(0.7625335606742392), 0.810138433899309],
 ['ridge', np.float64(0.7390303792535805), 0.885256022388444],
 ['LASSO', np.float64(0.05671614052695116), 1.5583699487861888],
 ['decision tree', np.float64(0.7971403667843827), 0.6753978221234475],
 ['random forest', np.float64(0.8853402172959411), 0.5325946130392877],
 ['extra trees', np.float64(0.873026621934823), 0.5435062205091261],
 ['gradient boosting', np.float64(0.8762358931566874), 0.5985112545680835],
 ['adaboost', np.float64(0.7531088291989377), 0.8540174312542974],
 ['mlp', np.float64(0.8080016682810769), 0.7566290020839748],
 ['xgboost', np.float64(0.8927843646619463), 0.539941539411034]]

In [24]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.885340,0.532595
10,xgboost,0.892784,0.539942
6,extra trees,0.873027,0.543506
7,gradient boosting,0.876236,0.598511
4,decision tree,0.797140,0.675398
9,mlp,0.808002,0.756629
1,svr,0.762534,0.810138
8,adaboost,0.753109,0.854017
2,ridge,0.739030,0.885256
0,linear_reg,0.739024,0.885367


| Column           | Data Used            | Scale | Interpretation                                      |
|------------------|----------------------|-------|----------------------------------------------------|
| r2               | y_transformed        | Log   | How well the model fits the curve of the data.     |
| mae              | np.expm1(y)          | Actual| How much the model is off by in real money.        |

<br>

##### **<span style="color:green">2. Ordinal Encoding + OneHotEncoding</span>**

<br>

###### **<span style="color:black">One Model</span>**

In [4]:
ordinal_cols = ['property_type', 'balcony', 'floor_category']
ohe_cols = ['sector', 'agePossession', 'furnishing_type']

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room', 'study room']),
        
        ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ohe_cols)
    ], 
    remainder='passthrough'
)

In [6]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [11]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
scores.mean(), scores.std()

(np.float64(0.8559454773763722), np.float64(0.02129949851967967))

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [13]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('ord', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [14]:
y_pred = pipeline.predict(X_test)
y_pred = np.expm1(y_pred)

In [15]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.6882154675557244

<br>

###### **<span style="color:black">ALL Models</span>**

In [16]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [17]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [18]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [19]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.887707,0.517886
10,xgboost,0.893431,0.522023
5,random forest,0.876594,0.531163
9,mlp,0.881817,0.554667
1,svr,0.885341,0.570169
7,gradient boosting,0.861635,0.621774
2,ridge,0.856862,0.683358
0,linear_reg,0.855945,0.688215
4,decision tree,0.803629,0.698055
8,adaboost,0.732869,0.889877


In [216]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.895030,0.466733
10,xgboost,0.896218,0.488796
5,random forest,0.891069,0.500622
9,mlp,0.879159,0.548368
7,gradient boosting,0.876766,0.569051
0,linear_reg,0.854615,0.649673
2,ridge,0.854673,0.652990
4,decision tree,0.807571,0.691544
8,adaboost,0.753009,0.814340
1,svr,0.769741,0.834124


<br>

##### **<span style="color:green">3. Ordinal Encoding + OneHotEncoding With PCA</span>**

<br>

###### **<span style="color:black">One Model</span>**

In [20]:
ordinal_cols = ['property_type', 'balcony', 'floor_category', 'furnishing_type']
ohe_cols = ['sector', 'agePossession']

In [27]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room', 'study room']),
        ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),        
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ohe_cols)
    ], 
    remainder='passthrough'
)

In [21]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), ordinal_cols),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False), ohe_cols)
    ], 
    remainder='passthrough'
)

In [28]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('regressor', LinearRegression())
])

In [29]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
scores.mean(), scores.std()

(np.float64(0.7664656054103705), np.float64(0.01926448225452247))

<br>

###### **<span style="color:black">ALL Models</span>**

In [30]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [31]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [32]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [33]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.843439,0.617558
10,xgboost,0.833644,0.670076
9,mlp,0.831915,0.675075
5,random forest,0.834321,0.678249
1,svr,0.821435,0.694081
7,gradient boosting,0.827645,0.723189
2,ridge,0.766506,0.844191
0,linear_reg,0.766466,0.844439
8,adaboost,0.689114,0.941686
4,decision tree,0.656662,1.017045


<br>

##### **<span style="color:green">4. Ordinal Encoding + OneHotEncoding + Target Encoder</span>**

<br>

###### **<span style="color:black">One Model</span>**

In [ ]:
!pip install category_encoders

In [ ]:
import category_encoders as ce

# Columns for different encoding techniques
ordinal_cols = ['property_type', 'balcony', 'furnishing_type', 'floor_category']
ohe_cols = ['agePossession']   # applying OHE to selected categorical feature
target_cols = ['sector']       # applying Target Encoding to high-cardinality feature

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        
        # Numerical features → Standard Scaling
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area','servant room', 'store room', 'study room']),
        
        # Ordinal Encoding → for general categorical features
        ('ord', OrdinalEncoder(), ordinal_cols),
        
        # One-Hot Encoding → for selected categorical feature
        ('ohe', OneHotEncoder(drop='first', sparse_output=False), ohe_cols),
        
        # Target Encoding → for high-cardinality feature (sector)
        ('target', ce.TargetEncoder(), target_cols)
        
    ], 
    remainder='passthrough'
)

In [36]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [37]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
scores.mean(),scores.std()

(np.float64(0.8293627899172942), np.float64(0.022073224927658952))

<br>

###### **<span style="color:black">ALL Models</span>**

In [38]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [39]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [40]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [41]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.902616,0.515282
6,extra trees,0.893438,0.527615
5,random forest,0.895945,0.541767
7,gradient boosting,0.887070,0.589498
1,svr,0.862992,0.646756
9,mlp,0.853745,0.674640
4,decision tree,0.805288,0.701948
2,ridge,0.829385,0.759271
0,linear_reg,0.829363,0.759591
8,adaboost,0.824092,0.769547


| Column           | Data Used            | Scale | Interpretation                                      |
|------------------|----------------------|-------|----------------------------------------------------|
| r2               | y_transformed        | Log   | How well the model fits the curve of the data.     |
| mae              | np.expm1(y)          | Actual| How much the model is off by in real money.        |

In [ ]:
model_df.sort_values(['mae'])       # Nitish Sir's

,name,r2,mae
5,random forest,0.900529,0.453631
6,extra trees,0.902299,0.457174
10,xgboost,0.900643,0.483409
7,gradient boosting,0.889074,0.509390
4,decision tree,0.830704,0.543814
9,mlp,0.852892,0.629715
8,adaboost,0.820024,0.681451
0,linear_reg,0.829522,0.713011
2,ridge,0.829536,0.713523
1,svr,0.782917,0.818851


<br>

#### **<span style="color:black">- Hyperparameter Tuning</span>**

<br>

###### **<span style="color:green">XGboost</span>**

In [92]:
param_grid_xgb = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_depth': [3, 5, 7],
    'regressor__learning_rate': [0.01, 0.05, 0.1],
    'regressor__subsample': [0.6, 0.8, 1.0],
    'regressor__colsample_bytree': [0.6, 0.8, 1.0],
    'regressor__gamma': [0, 0.1, 0.2],
    'regressor__reg_alpha': [0, 0.1, 1],
    'regressor__reg_lambda': [1, 1.5, 2]
}

In [93]:
# Columns for different encoding techniques
ordinal_cols = ['property_type', 'balcony', 'furnishing_type', 'floor_category']
ohe_cols = ['agePossession']   # applying OHE to selected categorical feature
target_cols = ['sector']       # applying Target Encoding to high-cardinality feature

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        
        # Numerical features → Standard Scaling
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area','servant room', 'store room', 'study room']),
        
        # Ordinal Encoding → for general categorical features
        ('ord', OrdinalEncoder(), ordinal_cols),
        
        # One-Hot Encoding → for selected categorical feature
        ('ohe', OneHotEncoder(drop='first', sparse_output=False), ohe_cols),
        
        # Target Encoding → for high-cardinality feature (sector)
        ('target', ce.TargetEncoder(), target_cols)
        
    ], 
    remainder='passthrough'
)

In [94]:
from xgboost import XGBRegressor

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state=42))
])

In [95]:
from sklearn.model_selection import KFold

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

In [96]:
from sklearn.model_selection import GridSearchCV

# Standard Grid Search focusing only on R2
search = GridSearchCV(
    pipeline,
    param_grid_xgb,   # your XGBoost grid
    cv=kfold,
    scoring='r2',     # Single metric scoring
    n_jobs=-1,
    verbose=4
)

search.fit(X, y_transformed)

Fitting 10 folds for each of 6561 candidates, totalling 65610 fits


,estimator,"Pipeline(step...te=42, ...))])"
,param_grid,"{'regressor__colsample_bytree': [0.6, 0.8, ...], 'regressor__gamma': [0, 0.1, ...], 'regressor__learning_rate': [0.01, 0.05, ...], 'regressor__max_depth': [3, 5, ...], ...}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,4
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('ord', ...), ...]"


In [97]:
final_pipe = search.best_estimator_

search.best_params_   # Best hyperparameters

{'regressor__colsample_bytree': 0.6,
 'regressor__gamma': 0,
 'regressor__learning_rate': 0.05,
 'regressor__max_depth': 7,
 'regressor__n_estimators': 300,
 'regressor__reg_alpha': 0,
 'regressor__reg_lambda': 1.5,
 'regressor__subsample': 0.8}

In [ ]:
search.best_score_    # Best CV R² score
# model is explaining 91% of the variance of the logarithm of the prices, not the actual prices.

np.float64(0.9102109394167428)

In [99]:
final_pipe.named_steps['regressor'].feature_importances_

array([0.11645829, 0.13171335, 0.17038524, 0.06578604, 0.01095534,
       0.01503963, 0.21731052, 0.02174354, 0.01794061, 0.01745996,
       0.01599508, 0.01808414, 0.01450089, 0.01395239, 0.15267493],
      dtype=float32)

In [104]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

# 1. Use cross_val_predict to get "Out-of-Fold" predictions for the WHOLE dataset
# This mimics the exact process GridSearchCV used to get that 0.91 score
y_pred_log = cross_val_predict(search.best_estimator_, X, y_transformed, cv=kfold)

# 2. Calculate R2 in Log Space (This should now match your ~0.91 score)
r2_log_cv = r2_score(y_transformed, y_pred_log)

# 3. Calculate MAE in Actual Space (This is your real-world error)
y_pred_actual = np.expm1(y_pred_log)
y_actual_real = np.expm1(y_transformed)
mae_actual_cv = mean_absolute_error(y_actual_real, y_pred_actual)

print(f"--- Tuned XGBoost (10-Fold CV Results) ---")
print(f"R2 (Log Space): {r2_log_cv:.6f}")
print(f"MAE (Actual Space): {mae_actual_cv:.6f}")

--- Tuned XGBoost (10-Fold CV Results) ---
R2 (Log Space): 0.910992
MAE (Actual Space): 0.473467


| Metric                     | Before Tuning (Baseline) | After Tuning (10-Fold CV) | Improvement  |
|---------------------------|--------------------------|----------------------------|--------------|
| $R^2$ Score (Log Space)   | 0.902616                 | 0.910992                   | +0.0083      |
| MAE (Actual Price)        | 0.515282                 | 0.473467                   | -0.0418      |
| Model Status              | 🥉 Audition Winner       | 🥇 Final Optimized Model   | Successful   |

<br>

#### **<span style="color:black">- Exporting the Model</span>**

Final Fit on all data

In [106]:
final_pipe = search.best_estimator_
final_pipe.fit(X,y_transformed)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('ord', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Save The Model

In [107]:
import pickle
with open('pipeline.pkl', 'wb') as f:
    pickle.dump(final_pipe, f)

In [108]:
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)

Load the Model

In [112]:
with open('pipeline.pkl', 'rb') as f:
    model = pickle.load(f)

Use the Model

In [128]:
# 4. Custom Prediction (Ensure ALL columns from training are present)
data = [['house', 'sector 102', 4, 3, '3+', 'New Property', 2750, 0, 0, 0, 'unfurnished', 'Low Floor']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
           'agePossession', 'built_up_area', 'servant room', 'store room', 
           'study room', 'furnishing_type', 'floor_category']

one_df = pd.DataFrame(data, columns=columns)

# Predict
prediction_log = model.predict(one_df)
prediction_real = np.expm1(prediction_log)
print(f"Predicted Price: {prediction_real[0]}")

Predicted Price: 3.1282477378845215
